# Session 2 Lab: Generative text classification

Classify the same Persian Telegram posts used in the embeddings exercise, this time with a generative model. Compare a GapGPT cloud model, direct GPT-5, and Gemma 3 through Ollama.

There is no gold standard. The goal is to reproduce one controlled workflow, inspect disagreement, and judge whether model rationales are grounded in the text.

## What stays fixed

All backends receive the same 25 posts, category definitions, system instructions, temperature, and JSON schema. Only the model route changes.

Important: Ollama in Google Colab runs inside a temporary Colab cloud machine. It is not the same as running a model on your own laptop.

In [ ]:
# Install packages — run once
!pip -q install pandas matplotlib requests openai ollama

In [ ]:
# Upload the dataset from the previous session
from google.colab import files
uploaded = files.upload()

In [ ]:
import io
import json
import time
from getpass import getpass

import matplotlib.pyplot as plt
import pandas as pd
import requests

DATA_FILE = 'telegram_policy_radar_4topics_fa.csv'
df = pd.read_csv(io.BytesIO(uploaded[DATA_FILE]))
print(f'Posts available: {len(df)}')
df.head(3)

## Select one backend

Run one backend first. Download its result file before switching to another backend. Use a small sample for class time and cost control.

In [ ]:
BACKEND = 'gapgpt'  # gapgpt, gpt5_openai, or ollama
SAMPLE_SIZE = 25
TEMPERATURE = 0

GAPGPT_BASE_URL = 'https://api.gapgpt.app/v1'
GAPGPT_MODEL = None  # None selects an available model automatically after you enter the key.
GAPGPT_FALLBACK_MODEL = 'gapgpt-deepseek-v3'  # Used only if the account cannot list models.
OPENAI_MODEL = 'gpt-5'
OLLAMA_MODEL = 'gemma3:4b'

sample = df.head(SAMPLE_SIZE).copy()
print('Backend:', BACKEND, '| Posts to classify:', len(sample))

## Shared classification specification

The prompt is written in English while the posts are Persian. This deliberately tests multilingual text analysis. Do not change the definitions without documenting the change.

In [ ]:
TOPICS = {
    'REG': 'Regulation and policy: laws, licences, rules, public institutions, councils, and digital-economy policy.',
    'AI': 'Artificial intelligence and data: AI, language models, data, algorithms, and AI applications.',
    'COMP': 'Competition and digital platforms: competition, monopoly, market power, platforms, e-commerce, and digital firms.',
    'TELCO': 'Telecom and digital infrastructure: internet access, telecom operators, fibre networks, 5G, connectivity, and communications infrastructure.'
}
SCHEMA = {
    'type': 'object',
    'properties': {
        'label': {'type': 'string', 'enum': list(TOPICS)},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'rationale': {'type': 'string'},
        'evidence': {'type': 'string'},
        'needs_review': {'type': 'boolean'}
    },
    'required': ['label', 'confidence', 'rationale', 'evidence', 'needs_review'],
    'additionalProperties': False
}
SYSTEM_PROMPT = '''You classify Persian news posts for a digital-market research team.
Use only the text of the post. Do not add outside facts.
Choose one label from REG, AI, COMP, TELCO.
Set needs_review to true when evidence is weak or two labels genuinely fit.
Keep rationale to one sentence. Evidence must be a short excerpt from the post.
Return only the requested JSON object.'''
print(TOPICS)

## Secure backend setup

Enter only the key required for the selected backend. Keep it private; never put a key in a notebook cell, shared file, or screenshot.

In [ ]:
gapgpt_key = None
openai_key = None
gapgpt_client = None
if BACKEND == 'gapgpt':
    from openai import OpenAI
    gapgpt_key = getpass('GapGPT API key (hidden): ')
    gapgpt_client = OpenAI(api_key=gapgpt_key, base_url=GAPGPT_BASE_URL)
    try:
        available_models = [model.id for model in gapgpt_client.models.list().data]
        if not available_models:
            raise ValueError('GapGPT returned an empty model list.')
        if GAPGPT_MODEL is None:
            preferred_models = ['gapgpt-deepseek-v3', 'deepseek-chat', 'gpt-4o-mini', 'gpt-4o', 'gapgpt-qwen-3.5']
            GAPGPT_MODEL = next((model for model in preferred_models if model in available_models), available_models[0])
        if GAPGPT_MODEL not in available_models:
            raise ValueError(f'{GAPGPT_MODEL} is not available for this GapGPT key. Choose one of: {available_models[:20]}')
        print('GapGPT model count:', len(available_models))
        print('Selected model:', GAPGPT_MODEL)
    except Exception as error:
        if GAPGPT_MODEL is None:
            GAPGPT_MODEL = GAPGPT_FALLBACK_MODEL
        print('Could not select from GapGPT models. The notebook will try:', GAPGPT_MODEL)
        print('Model-list message:', error)
elif BACKEND == 'gpt5_openai':
    openai_key = getpass('OpenAI API key (hidden): ')
elif BACKEND == 'ollama':
    print('No cloud API key is required. Run the Ollama setup cell next.')
else:
    raise ValueError('Choose a valid BACKEND value.')

## Ollama setup — run only for the ollama backend

This downloads a multi-gigabyte model into the temporary Colab runtime. It can be slow, and a free CPU runtime may be unsuitable for a full 25-post run.

In [ ]:
# Run only if BACKEND == 'ollama'
if BACKEND == 'ollama':
    !curl -fsSL https://ollama.com/install.sh | sh
    !nohup ollama serve > /tmp/ollama.log 2>&1 &
    time.sleep(5)
    !ollama pull gemma3:4b
    print('Ollama is ready.')

## API functions

Each backend receives the same instructions and is expected to return the same schema. Model output is validated before it enters the research table.

In [ ]:
def make_messages(post_text):
    return [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': 'Classify this post.\n\nPOST:\n' + post_text}
    ]

def validate_item(item):
    if item['label'] not in TOPICS:
        raise ValueError('Model returned an unknown label.')
    item['confidence'] = float(item['confidence'])
    if not 0 <= item['confidence'] <= 1:
        raise ValueError('Confidence is outside 0 to 1.')
    if not isinstance(item['needs_review'], bool):
        raise ValueError('needs_review must be true or false, not text.')
    return item

def parse_json(raw):
    text = raw.strip()
    fence = chr(96) * 3
    if text.startswith(fence):
        text = text.split('\n', 1)[1].rsplit(fence, 1)[0].strip()
    start, end = text.find('{'), text.rfind('}')
    if start == -1 or end == -1 or end < start:
        raise ValueError('No JSON object was found in the model response: ' + text[:240])
    return validate_item(json.loads(text[start:end + 1]))

def classify_gapgpt(post_text):
    response = gapgpt_client.chat.completions.create(
        model=GAPGPT_MODEL, messages=make_messages(post_text),
        temperature=TEMPERATURE)
    content = response.choices[0].message.content
    if not content:
        raise ValueError('GapGPT returned an empty response.')
    usage = {'input_tokens': getattr(response.usage, 'prompt_tokens', None),
             'output_tokens': getattr(response.usage, 'completion_tokens', None),
             'total_tokens': getattr(response.usage, 'total_tokens', None)}
    return parse_json(content), usage

def classify_gpt5(post_text):
    from openai import OpenAI
    client = OpenAI(api_key=openai_key)
    response = client.responses.create(
        model=OPENAI_MODEL, instructions=SYSTEM_PROMPT,
        input='Classify this post.\n\nPOST:\n' + post_text,
        reasoning={'effort': 'minimal'},
        text={'format': {'type': 'json_schema', 'name': 'post_classification',
                         'strict': True, 'schema': SCHEMA}})
    usage = {'input_tokens': getattr(response.usage, 'input_tokens', None),
             'output_tokens': getattr(response.usage, 'output_tokens', None)}
    return parse_json(response.output_text), usage

def classify_ollama(post_text):
    from ollama import chat
    response = chat(model=OLLAMA_MODEL, messages=make_messages(post_text),
                    format=SCHEMA, options={'temperature': TEMPERATURE})
    return parse_json(response.message.content), {}

## Run the controlled classification

Errors are recorded rather than silently discarded. Read those rows: format failures are part of evaluating a workflow.

In [ ]:
CLASSIFIERS = {'gapgpt': classify_gapgpt, 'gpt5_openai': classify_gpt5, 'ollama': classify_ollama}
MODELS = {'gapgpt': GAPGPT_MODEL, 'gpt5_openai': OPENAI_MODEL, 'ollama': OLLAMA_MODEL}
rows = []
for number, row in sample.reset_index(drop=True).iterrows():
    started = time.time()
    record = {'student_id': row['student_id'], 'month': row['month'], 'text_fa': row['text_fa'],
              'backend': BACKEND, 'model': MODELS[BACKEND]}
    try:
        output, usage = CLASSIFIERS[BACKEND](row['text_fa'])
        record.update(output)
        record['usage'] = json.dumps(usage)
        record['error'] = ''
    except Exception as error:
        record.update({'label': None, 'confidence': None, 'rationale': None, 'evidence': None,
                       'needs_review': True, 'usage': '', 'error': str(error)})
    record['seconds'] = round(time.time() - started, 2)
    rows.append(record)
    print(f'Completed {number + 1}/{len(sample)}')
results = pd.DataFrame(rows)
results.head()

## Inspect the findings

The chart shows model-assigned labels, not validated truth. Use individual rows to decide whether a rationale is grounded in the post.

In [ ]:
successful = results.dropna(subset=['label']).copy()
successful['confidence'] = pd.to_numeric(successful['confidence'], errors='coerce')
successful = successful.dropna(subset=['confidence'])
error_rows = results.loc[results['error'].ne(''), ['model', 'error', 'text_fa']].copy()
error_rows['error'] = error_rows['error'].astype(str).str.slice(0, 300)

print('Successful rows:', len(successful), '/', len(results))
print('Rows with errors:', len(error_rows))
print('Rows flagged for review:', successful['needs_review'].sum() if not successful.empty else 0)

if not error_rows.empty:
    print('First API or parsing errors — use these to diagnose the model ID, key, quota, or output format:')
    display(error_rows.head(5))

if successful.empty:
    print('No valid rows to chart. Fix the first error above, then rerun the classification cell.')
else:
    counts = successful['label'].value_counts().reindex(['REG', 'AI', 'COMP', 'TELCO'], fill_value=0)
    plt.figure(figsize=(8, 4.5))
    plt.bar(counts.index, counts.values, color=['#5B8FF9', '#61DDAA', '#65789B', '#F6BD16'])
    plt.title('Generative classification: ' + results.model.iloc[0])
    plt.ylabel('Posts')
    for i, value in enumerate(counts.values):
        plt.text(i, value + 0.2, str(value), ha='center')
    plt.show()
    display(successful.nlargest(5, 'confidence')[['label', 'confidence', 'rationale', 'evidence', 'text_fa']])
    display(successful.nsmallest(5, 'confidence')[['label', 'confidence', 'rationale', 'evidence', 'text_fa']])

## Compare models

Run the notebook once per backend and download each result file. Then upload two or three saved files below. This measures agreement, not accuracy.

In [ ]:
from google.colab import files
comparison_uploads = files.upload()

In [ ]:
frames = []
for filename, content in comparison_uploads.items():
    frame = pd.read_csv(io.BytesIO(content))
    frames.append(frame[['student_id', 'backend', 'model', 'label', 'confidence', 'needs_review']])
if frames:
    comparison = pd.concat(frames, ignore_index=True)
    label_matrix = comparison.pivot_table(index='student_id', columns='backend', values='label', aggfunc='first')
    display(label_matrix.head(10))
    if label_matrix.shape[1] > 1:
        label_matrix['all_agree'] = label_matrix.nunique(axis=1) == 1
        print('Full agreement rate:', round(label_matrix.all_agree.mean(), 2))
        display(label_matrix[~label_matrix.all_agree].head(10))
else:
    print('Upload two or more downloaded result files to compare them.')

## Download and document the run

Record the model, prompt version, date, temperature, sample size, token usage where available, and two examples requiring human review.

In [ ]:
OUTPUT_FILE = 'generative_classification_' + BACKEND + '.csv'
results.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
from google.colab import files
files.download(OUTPUT_FILE)

## Reference links

- [GapGPT](https://gapgpt.app) — API base URL: `https://api.gapgpt.app/v1`
- [OpenAI API quickstart](https://platform.openai.com/docs/quickstart/make-your-first-api-request)
- [Ollama structured outputs](https://docs.ollama.com/capabilities/structured-outputs)

Model availability, capabilities, quotas, and pricing can change. Check your provider dashboard immediately before the workshop.